# 01 · Run Qwen2.5-7B-Instruct over the texts

The model reads every text in `texts/` once, word by word, and this notebook saves what it saw.

| saved file | one row per | what's in it |
|---|---|---|
| `results/run/words/<text>.csv` | measured noun/verb | the word, the model's top 5 whole-word guesses (`candidates`) and their probabilities (`candidate_probs`), and the model-confidence values at that word |
| `results/run/tokens/<text>.csv` | token | the model-confidence values at every token |
| `results/run/run_log.csv` | text | how long each text took |

The model-confidence values follow LM-Polygraph's formulas; how every value is computed is explained in
`CONCLUSIONS.md`. Settings live in `qwen_config.py`.

**Runtime:** loading ~30 s; each new text 3–5 minutes (texts with saved results are skipped, so the notebook can be
stopped and re-run).

In [1]:
import os
import sys
import time
import warnings
from pathlib import Path

os.environ.setdefault("HF_HUB_OFFLINE", "1")   # the model is already in the Hugging Face cache
warnings.filterwarnings("ignore")
# make this folder's qwen_config.py and analysis/ importable, wherever Jupyter was started (e.g. the repo root)
QWEN7B_DIR = Path.cwd()
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "src/note_books/Qwen7B/qwen_config.py").exists():
        QWEN7B_DIR = folder / "src/note_books/Qwen7B"
        break
sys.path.insert(0, str(QWEN7B_DIR))

import analysis                                # also makes lca_core (repo root) importable
import pandas as pd
import torch
from IPython.display import display

from qwen_config import (GUESSES_PER_WORD, LOAD_IN_4BIT, MODEL_NAME, RUN_DIR, TEXTS_DIR, TOKENS_DIR,
                    WHOLE_WORD_GUESSES, WORDS_DIR)
from lca_core import LCAMeter, load_texts
from src.model.load_model import load_model

WORDS_DIR.mkdir(parents=True, exist_ok=True)
TOKENS_DIR.mkdir(parents=True, exist_ok=True)
texts = load_texts(TEXTS_DIR)
print(len(texts), "texts")

25 texts


## 1. Load the model

In [2]:
started = time.time()
tokenizer, model = load_model(MODEL_NAME, load_in_4bit=LOAD_IN_4BIT)
meter = LCAMeter(model=model, tokenizer=tokenizer)
print(f"loaded in {time.time() - started:.0f} s; "
      f"GPU memory in use: {torch.cuda.memory_allocated() / 2**30:.2f} GiB")

system check:
Python path: C:\Users\yoav7\miniconda3\python.exe
Torch version: 2.6.0+cu124
GPU: NVIDIA GeForce RTX 4060
PyTorch: 2.6.0+cu124
CUDA: 12.4
CUDA available: True
Capability: (8, 9)
this is include the following functions torch , torch.nn.functional as F, transformers.AutoTokenizer, transformers.AutoModelForCausalLM


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

loaded in 25 s; GPU memory in use: 5.45 GiB


## 2. Run every text
Texts that already have saved results are skipped.

In [4]:
log_path = RUN_DIR / "run_log.csv"
if log_path.exists():
    run_log = pd.read_csv(log_path)
else:
    run_log = pd.DataFrame(columns=["text", "measured words", "tokens", "seconds"])

for name, text in texts.items():
    words_path = WORDS_DIR / f"{name}.csv"
    tokens_path = TOKENS_DIR / f"{name}.csv"
    if words_path.exists() and tokens_path.exists():
        continue
    started = time.time()
    words = meter.run(text, k=GUESSES_PER_WORD, complete_words=WHOLE_WORD_GUESSES)
    words.to_csv(words_path, index=False)
    meter.last_tokens.to_csv(tokens_path, index=False)
    seconds = round(time.time() - started)
    new_row = {"text": name, "measured words": len(words), "tokens": len(meter.last_tokens), "seconds": seconds}
    run_log = pd.concat([run_log[run_log["text"] != name], pd.DataFrame([new_row])], ignore_index=True)
    run_log.to_csv(log_path, index=False)
    print(f"{name:34s} {len(words):4d} words  {len(meter.last_tokens):5d} tokens  {seconds:5d} s", flush=True)

print(f"{len(run_log)} texts measured, {run_log['seconds'].sum() / 3600:.1f} hours in total")
run_log

25 texts measured, 1.6 hours in total


,text,measured words,tokens,seconds
0,creative_gut_alice,156,896,331
1,creative_gut_grimm,183,828,246
2,creative_gut_oz,129,814,256
3,creative_gut_sherlock,172,813,191
4,creative_hp1,187,921,208
5,creative_hp2,195,944,212
6,creative_hp3,173,918,196
7,creative_hp4,170,908,189
8,creative_hp5,140,904,157
9,creative_hp6,180,921,198
